In [1]:
"""
심평원 데이터에서 구체적이고 전문적인 의료용어 1000개 추출
- 일반적인 용어 ('심실', '검사') 대신 구체적 용어 ('좌심실기능부전', '정밀면역검사') 추출
- 복합어, 전문 의학용어, 브랜드명 등 실제 의료진이 사용하는 용어 중심
"""

import pandas as pd
import re
from collections import Counter
from typing import List, Dict, Set

class SpecificMedicalTermExtractor:
    def __init__(self, excel_path: str):
        """심평원 데이터 로드"""
        self.df = pd.read_excel(excel_path)
        print(f"총 문서 수: {len(self.df):,}개")
    
    def extract_specific_medical_terms(self, text: str) -> List[str]:
        """구체적이고 전문적인 의료용어만 추출"""
        if not isinstance(text, str):
            return []
        
        specific_terms = set()
        
        # 1. 복합 질병명 (5글자 이상, 구체적)
        complex_diseases = re.findall(
            r'[가-힣]{3,}(?:증후군|병증|질환|부전|기능부전|장애|결핍증|과다증|협착증|폐색증|출혈증|혈전증|색전증|염증|감염증|중독증|위축증|비대증|이형성증|형성부전|무형성증)',
            text
        )
        for disease in complex_diseases:
            if 5 <= len(disease) <= 25 and '기타' not in disease and '상세불명' not in disease:
                specific_terms.add(disease)
        
        # 2. 구체적인 수술/시술명 (6글자 이상)
        complex_procedures = re.findall(
            r'[가-힣]{4,}(?:술|요법|치료법|삽입술|제거술|절제술|성형술|복원술|재건술|이식술|치환술|고정술|조성술|문합술|우회술|분리술|분할술|조영술|내시경술|천자술|절개술|봉합술|도관술)',
            text
        )
        for proc in complex_procedures:
            if 6 <= len(proc) <= 30 and '방법' not in proc:
                specific_terms.add(proc)
        
        # 3. 영문 복합 의학용어 (2단어 이상)
        english_complex = re.findall(
            r'\b[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,}){1,5}\b',
            text
        )
        for term in english_complex:
            if 8 <= len(term) <= 60 and 'Grade' not in term and 'Class' not in term:
                # 의학적 용어인지 확인
                medical_keywords = ['Syndrome', 'Disease', 'Disorder', 'Therapy', 'Treatment', 'Injection', 'Implant', 'Catheter', 'Membrane', 'Oxygenation', 'Transplantation']
                if any(keyword in term for keyword in medical_keywords):
                    specific_terms.add(term)
        
        # 4. 구체적인 검사명 (5글자 이상)
        specific_tests = re.findall(
            r'[가-힣]{3,}(?:검사|측정|진단|촬영|조영|분석|평가|관찰|모니터링|추적|스크리닝|생검|천자|도관|카테터|내시경)',
            text
        )
        for test in specific_tests:
            if 5 <= len(test) <= 25 and '일반' not in test and '기본' not in test and '단순' not in test:
                specific_terms.add(test)
        
        # 5. 약물 성분명 및 복합 약물명
        drug_components = re.findall(
            r'[가-힣]{3,}(?:나트륨|칼슘|마그네슘|염화물|황산염|인산염|아세트산|시트르산|글루콘산|락트산|말레산|타르타르산|옥살산|벤조산)',
            text
        )
        for drug in drug_components:
            if 5 <= len(drug) <= 30:
                specific_terms.add(drug)
        
        # 6. 해부학적 복합 구조 (4글자 이상)
        anatomy_complex = re.findall(
            r'[가-힣]{3,}(?:동맥|정맥|혈관|신경|근육|인대|연골|뼈|관절|막|선|조직|세포|수용체|결합조직|상피조직|결합체|복합체)',
            text
        )
        for anatomy in anatomy_complex:
            if 4 <= len(anatomy) <= 20 and anatomy not in ['혈관', '신경', '근육', '뼈']:  # 너무 일반적인 것 제외
                specific_terms.add(anatomy)
        
        # 7. 의료기기/치료재료 (4글자 이상)
        medical_devices = re.findall(
            r'[가-힣]{3,}(?:기|장치|시스템|카테터|스텐트|임플란트|보형물|삽입물|이식편|패치|메시|나사|핀|플레이트|로드|가이드|튜브|필터|센서|모니터)',
            text
        )
        for device in medical_devices:
            if 4 <= len(device) <= 25:
                specific_terms.add(device)
        
        # 8. 측정값/지표 (복합어)
        measurements = re.findall(
            r'[가-힣]{3,}(?:수치|지수|농도|압력|용량|속도|비율|분압|용적|함량|활성도|감수성|저항성|반응성|특이도|민감도)',
            text
        )
        for measure in measurements:
            if 4 <= len(measure) <= 25:
                specific_terms.add(measure)
        
        # 9. 영문 약물명 (생물학적 제제 포함)
        specific_drugs = re.findall(
            r'\b[A-Z][a-z]{4,}(?:mab|nib|pril|sartan|statin|mycin|cillin|cef|zole|dipine|olol|prazole|vir|nab|zumab|ximab|umab|lizumab|citabine|rubicin|platin|taxel|mide|pine|tide|ride|side|fide|kinib|tinib|zanib)\b',
            text
        )
        for drug in specific_drugs:
            if len(drug) >= 6:
                specific_terms.add(drug)
        
        # 10. 브랜드명 (더 구체적)
        brand_matches = re.findall(
            r'품명[:\s]*([가-힣A-Za-z0-9\s]{4,30})(?:정|주|캡슐|현탁액|주사제|겔|크림|연고|시럽|용액|분말|과립)',
            text
        )
        for brand in brand_matches:
            clean_brand = brand.strip()
            if 4 <= len(clean_brand) <= 25:
                specific_terms.add(clean_brand)
        
        # 11. 의료 약어와 복합어 조합
        medical_abbreviations = re.findall(
            r'\b(?:CT|MRI|PET|SPECT|ECG|EEG|EMG|ICD|VAD|ECMO|IABP|CABG|PTCA|CRT|CRRT|CVVH|SLED|BiVAD|LVAD|AICD)\s*[가-힣]{2,10}',
            text
        )
        for abbr_combo in medical_abbreviations:
            if len(abbr_combo) >= 5:
                specific_terms.add(abbr_combo)
        
        # 12. 복합 진료과목/전문분야
        specialty_complex = re.findall(
            r'[가-힣]{3,}(?:과|센터|클리닉|전문의|분야|영역|치료실|검사실|수술실)',
            text
        )
        for specialty in specialty_complex:
            if 4 <= len(specialty) <= 20:
                specific_terms.add(specialty)
        
        return list(specific_terms)
    
    def extract_top_1000_specific_terms(self, content_sample_size: int = 3000) -> List[tuple]:
        """상위 1000개 구체적 의료용어 추출"""
        print("구체적 의료용어 추출 시작...")
        
        all_terms = Counter()
        
        for index, row in self.df.iterrows():
            if index % 500 == 0:
                print(f"진행률: {index}/{len(self.df)} ({index/len(self.df)*100:.1f}%)")
            
            # 제목에서 추출 (가중치 3배)
            title = str(row.get('title', ''))
            title_terms = self.extract_specific_medical_terms(title)
            for term in title_terms:
                all_terms[term] += 3
            
            # 내용에서 추출 (샘플링)
            if index < content_sample_size:
                content = str(row.get('content', ''))
                content_terms = self.extract_specific_medical_terms(content)
                for term in content_terms:
                    all_terms[term] += 1
        
        print(f"총 추출된 구체적 의료용어: {len(all_terms):,}개")
        
        # 빈도 3회 이상인 용어만 선별하여 품질 보장
        quality_terms = [(term, freq) for term, freq in all_terms.items() if freq >= 3]
        quality_terms.sort(key=lambda x: x[1], reverse=True)
        
        print(f"품질 기준(3회 이상) 통과: {len(quality_terms):,}개")
        
        # 상위 1000개 반환
        return quality_terms[:1000]
    
    def categorize_specific_term(self, term: str) -> str:
        """구체적 의료용어 카테고리 분류"""
        if re.search(r'(?:증후군|병증|질환|부전|기능부전|장애|결핍증|과다증|협착증|폐색증|출혈증|혈전증|색전증|염증|감염증)$', term):
            return '복합질병명'
        elif re.search(r'(?:술|요법|치료법|삽입술|제거술|절제술|성형술|복원술|재건술|이식술|치환술|고정술|조성술|문합술|우회술)$', term):
            return '구체적수술시술'
        elif re.search(r'(?:검사|측정|진단|촬영|조영|분석|평가|관찰|모니터링|추적|스크리닝|생검|천자)$', term):
            return '전문검사진단'
        elif re.search(r'(?:동맥|정맥|혈관|신경|근육|인대|연골|뼈|관절|막|선|조직|세포|수용체)', term):
            return '상세해부구조'
        elif re.search(r'(?:기|장치|시스템|카테터|스텐트|임플란트|보형물|삽입물|이식편|패치|메시)$', term):
            return '의료기기치료재료'
        elif re.search(r'(?:나트륨|칼슘|마그네슘|염화물|황산염|인산염|아세트산)$', term):
            return '약물성분명'
        elif re.search(r'^[A-Z][a-z]+(?:mab|nib|pril|sartan|statin|mycin|cillin|cef|zole|dipine|olol|prazole)$', term):
            return '영문약물명'
        elif ' ' in term and re.match(r'^[A-Za-z\s]+$', term):
            return '영문복합의학용어'
        elif re.search(r'(?:수치|지수|농도|압력|용량|속도|비율|분압|용적|함량|활성도)$', term):
            return '측정값지표'
        else:
            return '기타전문용어'
    
    def save_specific_terms_csv(self, terms: List[tuple], filename: str = 'specific_medical_terms_1000.csv'):
        """구체적 의료용어를 CSV로 저장"""
        import csv
        
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['rank', 'term', 'frequency', 'category', 'length'])
            
            for rank, (term, freq) in enumerate(terms, 1):
                category = self.categorize_specific_term(term)
                writer.writerow([rank, term, freq, category, len(term)])
        
        print(f"구체적 의료용어 {len(terms)}개가 '{filename}'으로 저장되었습니다.")
    
    def analyze_specific_terms(self, terms: List[tuple]):
        """구체적 의료용어 통계 분석"""
        frequencies = [freq for _, freq in terms]
        categories = [self.categorize_specific_term(term) for term, _ in terms]
        lengths = [len(term) for term, _ in terms]
        
        print("\n=== 구체적 의료용어 통계 ===")
        print(f"총 용어 수: {len(terms):,}개")
        print(f"최고 빈도: {max(frequencies):,}회")
        print(f"최저 빈도: {min(frequencies):,}회")
        print(f"평균 빈도: {sum(frequencies)/len(frequencies):.1f}회")
        print(f"평균 길이: {sum(lengths)/len(lengths):.1f}글자")
        print(f"10회 이상: {len([f for f in frequencies if f >= 10]):,}개")
        print(f"5회 이상: {len([f for f in frequencies if f >= 5]):,}개")
        
        # 카테고리별 분포
        category_counts = Counter(categories)
        print(f"\n=== 카테고리별 분포 ===")
        for category, count in category_counts.most_common():
            print(f"{category}: {count}개")
        
        # 길이별 분포
        length_dist = Counter(lengths)
        print(f"\n=== 용어 길이별 분포 ===")
        for length in sorted(length_dist.keys()):
            if length_dist[length] >= 10:  # 10개 이상인 길이만 표시
                print(f"{length}글자: {length_dist[length]}개")



In [2]:

# 구체적 의료용어 추출기 생성
# /content/drive/MyDrive/CGINSIDE/보험심사/hira_datas.xlsx
# C:\Jimin\cg_suri_z-code360\code_analysis\raw_data\hira_datas.xlsx
data_path = r"C:\Jimin\cg_suri_z-code360\code_analysis\raw_data\hira_datas.xlsx"
extractor = SpecificMedicalTermExtractor(data_path)

# 상위 1000개 구체적 의료용어 추출
specific_terms = extractor.extract_top_1000_specific_terms()

# CSV로 저장
extractor.save_specific_terms_csv(specific_terms)

# 통계 분석
extractor.analyze_specific_terms(specific_terms)

# 상위 100개 출력
print(f"\n=== 구체적 의료용어 상위 100개 ===")
for rank, (term, freq) in enumerate(specific_terms[:100], 1):
    category = extractor.categorize_specific_term(term)
    print(f"{rank:3d}. {term} ({freq}회) - {category}")

print(f"\n=== 개선된 추출 결과 예시 ===")
print("기존: '심실' → 개선: '좌심실기능부전', '심실성빈맥증후군'")
print("기존: '검사' → 개선: '정밀면역검사', '유전자염기서열검사'")
print("기존: '수술' → 개선: '경피적관상동맥중재술', '로봇보조복강경수술'")

총 문서 수: 8,413개
구체적 의료용어 추출 시작...
진행률: 0/8413 (0.0%)
진행률: 500/8413 (5.9%)
진행률: 1000/8413 (11.9%)
진행률: 1500/8413 (17.8%)
진행률: 2000/8413 (23.8%)
진행률: 2500/8413 (29.7%)
진행률: 3000/8413 (35.7%)
진행률: 3500/8413 (41.6%)
진행률: 4000/8413 (47.5%)
진행률: 4500/8413 (53.5%)
진행률: 5000/8413 (59.4%)
진행률: 5500/8413 (65.4%)
진행률: 6000/8413 (71.3%)
진행률: 6500/8413 (77.3%)
진행률: 7000/8413 (83.2%)
진행률: 7500/8413 (89.1%)
진행률: 8000/8413 (95.1%)
총 추출된 구체적 의료용어: 4,616개
품질 기준(3회 이상) 통과: 2,643개
구체적 의료용어 1000개가 'specific_medical_terms_1000.csv'으로 저장되었습니다.

=== 구체적 의료용어 통계 ===
총 용어 수: 1,000개
최고 빈도: 354회
최저 빈도: 4회
평균 빈도: 13.9회
평균 길이: 7.4글자
10회 이상: 356개
5회 이상: 958개

=== 카테고리별 분포 ===
전문검사진단: 197개
구체적수술시술: 191개
기타전문용어: 189개
의료기기치료재료: 158개
상세해부구조: 95개
복합질병명: 67개
영문약물명: 54개
영문복합의학용어: 38개
측정값지표: 10개
약물성분명: 1개

=== 용어 길이별 분포 ===
4글자: 140개
5글자: 177개
6글자: 213개
7글자: 149개
8글자: 89개
9글자: 61개
10글자: 53개
11글자: 38개
12글자: 29개

=== 구체적 의료용어 상위 100개 ===
  1. 조혈모세포 (354회) - 상세해부구조
  2. 정밀면역검사 (256회) - 전문검사진단
  3. 신의료기 (205회) - 의료기기치료재료
  4. 솔리

In [3]:
"""
전체 심평원 데이터 8,413개 문서에서 의료용어 완전 추출
- 모든 문서의 제목과 내용 전체 분석
- 3,000-5,000개 의료용어 예상
- 정확한 빈도 계산 및 품질 보장
"""

import pandas as pd
import re
from collections import Counter
from typing import List, Dict, Set
import time
import json

class FullMedicalTermExtractor:
    def __init__(self, excel_path: str):
        """심평원 전체 데이터 로드"""
        print("📊 심평원 데이터 로드 중...")
        self.df = pd.read_excel(excel_path)
        print(f"✅ 총 {len(self.df):,}개 문서 로드 완료")
        
        # 진행상황 추적
        self.processed_docs = 0
        self.start_time = None
        
    def extract_comprehensive_medical_terms(self, text: str) -> List[str]:
        """포괄적이고 정밀한 의료용어 추출"""
        if not isinstance(text, str) or len(text) < 3:
            return []
        
        medical_terms = set()
        
        # 1. 복합 질병명 및 증후군 (더 포괄적)
        disease_patterns = [
            r'[가-힣]{3,}(?:증후군|병증|질환|부전|기능부전|장애|결핍증|과다증|협착증|폐색증|출혈증|혈전증|색전증|염증|감염증|중독증|위축증|비대증|이형성증|형성부전|무형성증|이상증|손상|파열|천공|궤양|미란|용종|낭종|종괴)',
            r'[가-힣]{4,}(?:병|암|종|염|증|루|관|체|혈증|뇨증|혈뇨|뇨당|혈당|혈압)'
        ]
        
        for pattern in disease_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 4 <= len(match) <= 30 and '기타' not in match and '상세불명' not in match:
                    medical_terms.add(match)
        
        # 2. 수술 및 시술명 (매우 포괄적)
        procedure_patterns = [
            r'[가-힣]{4,}(?:술|요법|치료법|처치|삽입술|제거술|절제술|성형술|복원술|재건술|이식술|치환술|고정술|조성술|문합술|우회술|분리술|분할술|조영술|내시경술|천자술|절개술|봉합술|도관술|카테터술|스텐트술)',
            r'[가-힣]{3,}(?:마취|진정|수혈|투석|관류|순환|보조|지지|견인|압박|냉각|가온|전기|레이저|초음파|고주파|냉동|열치료)'
        ]
        
        for pattern in procedure_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 5 <= len(match) <= 35 and '방법' not in match and '일반' not in match:
                    medical_terms.add(match)
        
        # 3. 영문 의학용어 (2-5단어 복합어 포함)
        english_medical = re.findall(
            r'\b[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,}){1,4}\b',
            text
        )
        
        medical_keywords = [
            'Syndrome', 'Disease', 'Disorder', 'Therapy', 'Treatment', 'Surgery', 'Procedure',
            'Injection', 'Implant', 'Catheter', 'Stent', 'Graft', 'Transplant', 'Ablation',
            'Membrane', 'Oxygenation', 'Ventilation', 'Circulation', 'Perfusion', 'Monitoring',
            'Analysis', 'Assay', 'Screening', 'Diagnosis', 'Imaging', 'Tomography', 'Resonance',
            'Cardiomyopathy', 'Neuropathy', 'Nephropathy', 'Retinopathy', 'Arthropathy'
        ]
        
        for term in english_medical:
            if 8 <= len(term) <= 60:
                if any(keyword in term for keyword in medical_keywords):
                    medical_terms.add(term)
        
        # 4. 정밀 검사 및 진단명
        test_patterns = [
            r'[가-힣]{4,}(?:검사|측정|진단|촬영|조영|분석|평가|관찰|모니터링|추적|스크리닝|생검|천자|도관|카테터|내시경|현미경|전기|방사선|동위원소|초음파|자기공명)',
            r'[가-힣]{3,}(?:면역|유전자|염색체|단백질|효소|호르몬|대사|생화학|미생물|바이러스|세균|진균|기생충)(?:검사|분석|측정)'
        ]
        
        for pattern in test_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 5 <= len(match) <= 30 and '일반' not in match and '기본' not in match:
                    medical_terms.add(match)
        
        # 5. 약물 및 성분명 (확장)
        drug_patterns = [
            r'[가-힣]{4,}(?:나트륨|칼슘|마그네슘|칼륨|철|아연|구리|염화물|황산염|인산염|탄산염|아세트산|시트르산|글루콘산|락트산|말레산|타르타르산|옥살산|벤조산|살리실산)',
            r'[가-힣]{4,}(?:주사제|정제|캡슐|현탁액|용액|겔|크림|연고|시럽|분말|과립|좌제|패치|스프레이|흡입제|점안제|점비제|점이제)'
        ]
        
        for pattern in drug_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 5 <= len(match) <= 30:
                    medical_terms.add(match)
        
        # 6. 상세 해부학적 구조
        anatomy_patterns = [
            r'[가-힣]{3,}(?:동맥|정맥|혈관|모세혈관|림프관|신경|신경근|신경절|신경총|근육|골격근|심근|평활근)',
            r'[가-힣]{3,}(?:인대|건|연골|활막|관절|관절낭|추간판|척수|뇌간|소뇌|대뇌|뇌실|뇌막)',
            r'[가-힣]{3,}(?:선|관|낭|막|조직|세포|수용체|효소|단백질|호르몬|신경전달물질)'
        ]
        
        for pattern in anatomy_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 4 <= len(match) <= 25 and match not in ['혈관', '신경', '근육', '뼈', '관절']:
                    medical_terms.add(match)
        
        # 7. 의료기기 및 치료재료 (확장)
        device_patterns = [
            r'[가-힣]{3,}(?:기|장치|시스템|기계|펌프|모니터|센서|감지기|측정기|분석기|처리기)',
            r'[가-힣]{3,}(?:카테터|튜브|도관|캐뉼라|스텐트|임플란트|보형물|삽입물|이식편|인공|로봇)',
            r'[가-힣]{3,}(?:패치|메시|나사|핀|플레이트|로드|와이어|클립|코일|스프링|필터|막)'
        ]
        
        for pattern in device_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 4 <= len(match) <= 30:
                    medical_terms.add(match)
        
        # 8. 생물학적 제제 및 첨단 약물 (확장)
        biologic_patterns = [
            r'\b[A-Z][a-z]{4,}(?:mab|nib|pril|sartan|statin|mycin|cillin|cef|zole|dipine|olol|prazole|vir|nab|zumab|ximab|umab|lizumab|citabine|rubicin|platin|taxel|mide|pine|tide|ride|side|fide|kinib|tinib|zanib|afenib|tinib)\b',
            r'\b[A-Z][a-z]{3,}(?:alfa|beta|gamma|delta|omega)\b'
        ]
        
        for pattern in biologic_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if len(match) >= 6:
                    medical_terms.add(match)
        
        # 9. 브랜드명 및 제품명 (확장)
        brand_patterns = [
            r'품명[:\s]*([가-힣A-Za-z0-9\s\-]{4,30})(?:정|주|캡슐|현탁액|주사제|겔|크림|연고|시럽|용액|분말|과립|좌제|패치|스프레이)',
            r'상품명[:\s]*([가-힣A-Za-z0-9\s\-]{4,30})(?:정|주|캡슐)',
            r'제품명[:\s]*([가-힣A-Za-z0-9\s\-]{4,30})(?:정|주|캡슐)'
        ]
        
        for pattern in brand_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                clean_brand = match.strip()
                if 4 <= len(clean_brand) <= 25 and not re.search(r'^\d+$', clean_brand):
                    medical_terms.add(clean_brand)
        
        # 10. 측정값 및 지표 (확장)
        measurement_patterns = [
            r'[가-힣]{3,}(?:수치|지수|농도|압력|용량|용적|속도|비율|분압|함량|활성도|감수성|저항성|반응성|특이도|민감도|정확도|정밀도)',
            r'[가-힣]{3,}(?:계수|분율|백분율|지표|인덱스|스코어|점수|등급|단계|정도|강도|밀도|점도|탄성|경도)'
        ]
        
        for pattern in measurement_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 4 <= len(match) <= 25:
                    medical_terms.add(match)
        
        # 11. 의료 약어와 복합어
        abbreviation_patterns = [
            r'\b(?:CT|MRI|PET|SPECT|ECG|EEG|EMG|ICD|VAD|ECMO|IABP|CABG|PTCA|CRT|CRRT|CVVH|SLED|BiVAD|LVAD|AICD|TAVI|TAVR|EVAR|TEVAR)\s*[가-힣]{2,15}',
            r'[가-힣]{2,15}\s*(?:CT|MRI|PET|SPECT|ECG|EEG|EMG)'
        ]
        
        for pattern in abbreviation_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 5 <= len(match) <= 25:
                    medical_terms.add(match.strip())
        
        # 12. 특수 의료 용어 (ICD, CPT 코드 관련)
        special_patterns = [
            r'[가-힣]{4,}(?:분류|코드|번호|항목|범주|군|류|계열|체계|기준|지침|규정|방법|절차|과정|단계|프로토콜)',
            r'[가-힣]{3,}(?:합병증|부작용|이상반응|금기|주의|경고|위험|안전성|유효성|적응증|적응|승인|허가|등록)'
        ]
        
        for pattern in special_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if 4 <= len(match) <= 20:
                    medical_terms.add(match)
        
        return list(medical_terms)
    
    def update_progress(self, current: int, total: int):
        """진행상황 업데이트"""
        if self.start_time is None:
            self.start_time = time.time()
        
        if current % 500 == 0 or current == total:
            elapsed = time.time() - self.start_time
            progress = current / total * 100
            
            if current > 0:
                eta = elapsed * (total - current) / current
                eta_min = int(eta // 60)
                eta_sec = int(eta % 60)
                
                print(f"📈 진행률: {current:,}/{total:,} ({progress:.1f}%) | "
                      f"경과: {int(elapsed//60)}분 {int(elapsed%60)}초 | "
                      f"예상 잔여: {eta_min}분 {eta_sec}초")
            else:
                print(f"📈 진행률: {current:,}/{total:,} ({progress:.1f}%)")
    
    def extract_all_medical_terms(self) -> Dict[str, int]:
        """전체 문서에서 의료용어 추출"""
        print(f"\n🚀 전체 {len(self.df):,}개 문서 분석 시작...")
        print("⏰ 예상 소요 시간: 15-30분")
        
        all_terms = Counter()
        
        for index, row in self.df.iterrows():
            self.update_progress(index, len(self.df))
            
            # 제목에서 추출 (가중치 3배)
            title = str(row.get('title', ''))
            if title and title != 'nan':
                title_terms = self.extract_comprehensive_medical_terms(title)
                for term in title_terms:
                    all_terms[term] += 3
            
            # 내용에서 추출 (가중치 1배)
            content = str(row.get('content', ''))
            if content and content != 'nan' and len(content) > 10:
                content_terms = self.extract_comprehensive_medical_terms(content)
                for term in content_terms:
                    all_terms[term] += 1
        
        # 최종 진행률 업데이트
        self.update_progress(len(self.df), len(self.df))
        
        total_time = time.time() - self.start_time
        print(f"\n✅ 전체 분석 완료! 총 소요시간: {int(total_time//60)}분 {int(total_time%60)}초")
        print(f"📊 추출된 고유 의료용어: {len(all_terms):,}개")
        
        return dict(all_terms)
    
    def filter_quality_terms(self, all_terms: Dict[str, int], min_frequency: int = 2) -> List[tuple]:
        """품질 기준에 따라 의료용어 필터링"""
        print(f"\n🔍 품질 필터링 중... (최소 빈도: {min_frequency}회)")
        
        # 빈도 기준 필터링
        quality_terms = [(term, freq) for term, freq in all_terms.items() 
                        if freq >= min_frequency and 3 <= len(term) <= 50]
        
        # 빈도순 정렬
        quality_terms.sort(key=lambda x: x[1], reverse=True)
        
        print(f"✅ 품질 기준 통과: {len(quality_terms):,}개")
        
        # 빈도별 분포 분석
        freq_dist = Counter([freq for _, freq in quality_terms])
        print(f"\n📈 빈도 분포:")
        for freq in sorted(freq_dist.keys(), reverse=True)[:10]:
            if freq >= 5:
                print(f"  {freq}회: {freq_dist[freq]}개 용어")
        
        return quality_terms
    
    def save_comprehensive_results(self, terms: List[tuple], filename: str = 'comprehensive_medical_terms.csv'):
        """전체 결과를 CSV로 저장"""
        import csv
        
        print(f"\n💾 결과 저장 중: '{filename}'")
        
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['rank', 'term', 'frequency', 'category', 'length', 'tier'])
            
            for rank, (term, freq) in enumerate(terms, 1):
                category = self.categorize_comprehensive_term(term)
                length = len(term)
                
                # 티어 분류
                if freq >= 50:
                    tier = "Tier1_초고빈도"
                elif freq >= 20:
                    tier = "Tier2_고빈도"  
                elif freq >= 10:
                    tier = "Tier3_중빈도"
                elif freq >= 5:
                    tier = "Tier4_중저빈도"
                else:
                    tier = "Tier5_저빈도"
                
                writer.writerow([rank, term, freq, category, length, tier])
        
        print(f"✅ {len(terms):,}개 의료용어가 저장되었습니다.")
    
    def categorize_comprehensive_term(self, term: str) -> str:
        """포괄적 의료용어 카테고리 분류"""
        if re.search(r'(?:증후군|병증|질환|부전|기능부전|장애|결핍증|과다증|협착증|폐색증|출혈증|혈전증|색전증|염증|감염증|병|암|종)$', term):
            return '질병및증후군'
        elif re.search(r'(?:술|요법|치료법|처치|삽입술|제거술|절제술|성형술|복원술|재건술|이식술|치환술|고정술|조성술|문합술|우회술|마취|수혈|투석)$', term):
            return '수술및시술'
        elif re.search(r'(?:검사|측정|진단|촬영|조영|분석|평가|관찰|모니터링|추적|스크리닝|생검|천자|면역|유전자).*(?:검사|분석|측정)$', term):
            return '검사및진단'
        elif re.search(r'(?:동맥|정맥|혈관|신경|근육|인대|연골|뼈|관절|막|선|조직|세포|수용체)', term):
            return '해부및생리'
        elif re.search(r'(?:기|장치|시스템|카테터|스텐트|임플란트|보형물|삽입물|이식편|인공|로봇|펌프|모니터)', term):
            return '의료기기'
        elif re.search(r'(?:나트륨|칼슘|마그네슘|주사제|정제|캡슐|현탁액|용액|겔|크림|연고)$', term):
            return '약물및성분'
        elif re.search(r'^[A-Z][a-z]+(?:mab|nib|pril|sartan|statin|mycin|cillin|cef|zole|dipine|olol|prazole)$', term):
            return '생물학적제제'
        elif ' ' in term and re.match(r'^[A-Za-z\s]+$', term):
            return '영문의학용어'
        elif re.search(r'(?:수치|지수|농도|압력|용량|속도|비율|분압|용적|함량|활성도|감수성|저항성|반응성)$', term):
            return '측정값지표'
        else:
            return '기타의료용어'
    
    def analyze_comprehensive_results(self, terms: List[tuple]):
        """전체 결과 통계 분석"""
        frequencies = [freq for _, freq in terms]
        categories = [self.categorize_comprehensive_term(term) for term, _ in terms]
        lengths = [len(term) for term, _ in terms]
        
        print(f"\n📊 === 전체 의료용어 분석 결과 ===")
        print(f"총 의료용어: {len(terms):,}개")
        print(f"최고 빈도: {max(frequencies):,}회")
        print(f"최저 빈도: {min(frequencies):,}회") 
        print(f"평균 빈도: {sum(frequencies)/len(frequencies):.1f}회")
        print(f"평균 길이: {sum(lengths)/len(lengths):.1f}글자")
        
        # 빈도별 분포
        freq_ranges = [
            ("100회 이상", len([f for f in frequencies if f >= 100])),
            ("50-99회", len([f for f in frequencies if 50 <= f < 100])),
            ("20-49회", len([f for f in frequencies if 20 <= f < 50])),
            ("10-19회", len([f for f in frequencies if 10 <= f < 20])),
            ("5-9회", len([f for f in frequencies if 5 <= f < 10])),
            ("2-4회", len([f for f in frequencies if 2 <= f < 5]))
        ]
        
        print(f"\n📈 빈도별 분포:")
        for freq_range, count in freq_ranges:
            if count > 0:
                print(f"  {freq_range}: {count:,}개")
        
        # 카테고리별 분포
        category_counts = Counter(categories)
        print(f"\n🏷️  카테고리별 분포:")
        for category, count in category_counts.most_common():
            print(f"  {category}: {count:,}개")



In [ ]:
# 실행 함수
def run_comprehensive_extraction():
    """전체 의료용어 추출 실행"""
    print("🏥 === 전체 심평원 데이터 의료용어 추출 ===")
    print("📋 분석 대상: 8,413개 문서 (제목 + 내용 전체)")
    print("🎯 목표: 3,000-5,000개 의료용어 추출")
    print("⏰ 예상 시간: 15-30분\n")
    
    # 추출기 생성
    extractor = FullMedicalTermExtractor('hira_datas.xlsx')
    
    # 전체 의료용어 추출
    all_terms = extractor.extract_all_medical_terms()
    
    # 품질 필터링
    quality_terms = extractor.filter_quality_terms(all_terms, min_frequency=2)
    
    # 결과 저장
    extractor.save_comprehensive_results(quality_terms)
    
    # 통계 분석
    extractor.analyze_comprehensive_results(quality_terms)
    
    # 상위 100개 출력
    print(f"\n🔝 상위 100개 의료용어:")
    for rank, (term, freq) in enumerate(quality_terms[:100], 1):
        category = extractor.categorize_comprehensive_term(term)
        print(f"{rank:3d}. {term} ({freq}회) - {category}")
    
    print(f"\n🎉 전체 분석 완료!")
    print(f"📁 결과 파일: 'comprehensive_medical_terms.csv'")
    print(f"📊 총 추출 용어: {len(quality_terms):,}개")
    
    return quality_terms

if __name__ == "__main__":
    # 전체 추출 실행
    results = run_comprehensive_extraction()